# A1 — OLS Cross-Section

**Famiglia A** | Una riga per contatore | Baseline della tesi

```
pct_silente_i = β₀ + β·X_i + ε_i
```
- Variabile dipendente: proporzione di finestre silenti [0, 1]
- Include variabili statiche + aggregate dinamiche
- **Prerequisito**: eseguire `00_preparazione_dati/01_Preparazione_dati_v2.ipynb` per generare i parquet

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 1 — Setup: mount Drive + clone repo + caricamento parquet
# ══════════════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount("/content/drive")

import subprocess, os, sys

REPO_URL  = "https://github.com/swaggincoffee-bit/Tesi.git"
REPO_PATH = "/content/Tesi"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
sys.path.insert(0, REPO_PATH)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")

from src.config import (
    COL_DATA_INST, COL_ANNO_COSTR, COL_CONSUMO,
    COL_MODELLO, COL_TECN_COM, COL_COSTRUTTORE,
    FNAME_CS,
)

PROC = "/content/drive/MyDrive/Contatori/OUTPUT/"
OUT  = "/content/drive/MyDrive/Contatori/OUTPUT/"

df_cs = pd.read_parquet(PROC + FNAME_CS)

print(f"df_cs caricato: {df_cs.shape}")
display(df_cs.head(3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
df_cs caricato: (226702, 53)


,04IND - Provincia (cod),04IND - Comune (cod),04IND - CAP (cod),03IMPS - STec - Accessibile (cod),03IMPS - PDR/POD - Esterno (cod),03IMPS - Stato telegestione (cod),03IMPS - Operandi - Cons.Anno PDR (desc),11MDEF - Installazione(dta),11MDEF - Costruttore (cod),11MDEF - Anno Costruzione (cod),...,gg_da_installazione,firmware_num,gg_da_ult_com,gg_da_ult_mis,costruttore,modello,tecnologia,comune,accessibile,telegestione
0,Reggio nell'Emilia,Albinea,42020,ACCESSIBILE,15441000000750,GT,916,17.12.2024,METERSIT,2024,...,370,732,-40.0,-39.0,METERSIT,G4_NBIOT,NBIOT,Albinea,ACCESSIBILE,GT
1,Reggio nell'Emilia,Albinea,42020,ACCESSIBILE,15441000001565,GT,2182,22.10.2024,METERSIT,2024,...,426,732,-44.0,-43.0,METERSIT,G6_NBIOT,NBIOT,Albinea,ACCESSIBILE,GT
2,Reggio nell'Emilia,Albinea,42020,ACCESSIBILE,15441000002472,GT,156,11.08.2017,METERSIT,2017,...,3055,400,-45.0,-43.0,METERSIT,DOMUS NEXT G4,GPRS,Albinea,ACCESSIBILE,GT


In [2]:
# CELLA 2 - Feature engineering

from sklearn.preprocessing import LabelEncoder

df_lett_re = pd.read_parquet(PROC + "df_lett_re.parquet")
ref_date = pd.to_datetime(df_lett_re["data_lettura"]).max()
del df_lett_re

# Categoriche -> encoded
for short, col in {"costruttore_enc":  "costruttore",
                   "modello_enc":      "modello",
                   "tecnologia_enc":   "tecnologia",
                   "comune_enc":       "comune",
                   "accessibile_enc":  "accessibile",
                   "telegestione_enc": "telegestione"}.items():
    df_cs[short] = LabelEncoder().fit_transform(df_cs[col].fillna("N/A"))

FEATURES = [
    "anni_da_costruzione",
    "consumo_annuo",
    "gg_da_installazione",
    "firmware_num",
    "gg_da_ult_com",
    "gg_da_ult_mis",
    "costruttore_enc",
    "modello_enc",
    "tecnologia_enc",
    "comune_enc",
    "accessibile_enc",
    "telegestione_enc",
    "pct_err",
    *[f"diag_bit_{i:02d}" for i in range(16)],
]
TARGET = "pct_silente"

df_model = df_cs[FEATURES + [TARGET]].dropna()
print(f"Osservazioni: {len(df_model):,}  (NaN scartati: {len(df_cs)-len(df_model):,})")
display(df_model[FEATURES].describe().round(2))

Osservazioni: 219,330  (NaN scartati: 7,372)


,anni_da_costruzione,consumo_annuo,gg_da_installazione,firmware_num,gg_da_ult_com,gg_da_ult_mis,costruttore_enc,modello_enc,tecnologia_enc,comune_enc,...,diag_bit_06,diag_bit_07,diag_bit_08,diag_bit_09,diag_bit_10,diag_bit_11,diag_bit_12,diag_bit_13,diag_bit_14,diag_bit_15
count,219330.00,219330.00,219330.00,219330.00,219330.00,219330.00,219330.00,219330.00,219330.00,219330.00,...,219330.00,219330.00,219330.0,219330.00,219330.00,219330.00,219330.00,219330.00,219330.00,219330.00
mean,5.65,845.58,2146.47,503.58,-23.62,-22.60,4.49,6.74,2.35,22.43,...,0.00,0.00,0.0,0.00,0.01,0.00,0.00,0.03,0.03,0.01
std,2.77,876.57,1005.79,1075.01,75.43,76.69,1.47,4.02,1.17,9.38,...,0.01,0.02,0.0,0.01,0.09,0.03,0.03,0.18,0.16,0.12
min,0.00,0.00,-9.00,138.00,-45.00,-44.00,1.00,0.00,0.00,0.00,...,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,4.00,278.00,1417.00,500.00,-45.00,-43.00,4.00,6.00,2.00,16.00,...,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,7.00,708.00,2510.00,500.00,-44.00,-43.00,4.00,6.00,3.00,27.00,...,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00
75%,8.00,1193.00,2884.00,503.00,-43.00,-42.00,4.00,11.00,3.00,27.00,...,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00
max,24.00,51000.00,3975.00,250801.00,2031.00,688.00,7.00,15.00,3.00,41.00,...,1.00,1.00,1.0,1.00,1.00,1.00,1.00,1.00,1.00,1.00


In [3]:
print(df_cs.columns.tolist())

['04IND - Provincia (cod)', '04IND - Comune (cod)', '04IND - CAP (cod)', '03IMPS - STec - Accessibile (cod)', '03IMPS - PDR/POD - Esterno (cod)', '03IMPS - Stato telegestione (cod)', '03IMPS - Operandi - Cons.Anno PDR (desc)', '11MDEF - Installazione(dta)', '11MDEF - Costruttore (cod)', '11MDEF - Anno Costruzione (cod)', '11MDEF - Materiale (cod)', '11MDEF - Numero Serie (cod)', 'MEAS - PDR (conteggio)', '_key', 'MATRICOLA CONTATORE', 'PDR', 'MODELLO CONTATORE', 'VERSIONE FIRMWARE', 'Tecnologia di comunicazione', 'Data ultima comunicazione', 'Data ultima misura', 'pct_silente', 'n_finestre', 'silente_prevalente', 'pct_err', 'diag_bit_00', 'diag_bit_01', 'diag_bit_02', 'diag_bit_03', 'diag_bit_04', 'diag_bit_05', 'diag_bit_06', 'diag_bit_07', 'diag_bit_08', 'diag_bit_09', 'diag_bit_10', 'diag_bit_11', 'diag_bit_12', 'diag_bit_13', 'diag_bit_14', 'diag_bit_15', 'anni_da_costruzione', 'consumo_annuo', 'gg_da_installazione', 'firmware_num', 'gg_da_ult_com', 'gg_da_ult_mis', 'costruttore', 

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 3 — OLS con statsmodels
# ══════════════════════════════════════════════════════════════════════════════

X = sm.add_constant(df_model[FEATURES])
y = df_model[TARGET]

model = sm.OLS(y, X).fit(cov_type="HC3")  # errori robusti eteroschedasticità
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:            pct_silente   R-squared:                       0.388
Model:                            OLS   Adj. R-squared:                  0.388
Method:                 Least Squares   F-statistic:                     6936.
Date:                Sun, 10 May 2026   Prob (F-statistic):               0.00
Time:                        22:26:12   Log-Likelihood:             2.2941e+05
No. Observations:              219330   AIC:                        -4.588e+05
Df Residuals:                  219300   BIC:                        -4.584e+05
Df Model:                          29                                         
Covariance Type:                  HC3                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   0.9201    

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 3 — OLS con statsmodels
# ══════════════════════════════════════════════════════════════════════════════

X = sm.add_constant(df_model[FEATURES])
y = df_model[TARGET]

model = sm.OLS(y, X).fit(cov_type="HC3")  # errori robusti eteroschedasticità
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:            pct_silente   R-squared:                       0.388
Model:                            OLS   Adj. R-squared:                  0.388
Method:                 Least Squares   F-statistic:                     6936.
Date:                Sun, 10 May 2026   Prob (F-statistic):               0.00
Time:                        22:26:14   Log-Likelihood:             2.2941e+05
No. Observations:              219330   AIC:                        -4.588e+05
Df Residuals:                  219300   BIC:                        -4.584e+05
Df Model:                          29                                         
Covariance Type:                  HC3                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   0.9201    

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 5 — Salvataggio risultati
# ══════════════════════════════════════════════════════════════════════════════

risultati = model.summary2().tables[1].reset_index()
risultati.columns = ["variabile", "coef", "std_err", "t", "p_value", "ci_low", "ci_high"]
risultati.to_csv(OUT + "A1_coefficienti_OLS.csv", index=False)

print("✅ Salvati:")
print("  A1_diagnostica_residui.png")
print("  A1_coefficienti_OLS.csv")
display(risultati)

✅ Salvati:
  A1_diagnostica_residui.png
  A1_coefficienti_OLS.csv


,variabile,coef,std_err,t,p_value,ci_low,ci_high
0,const,9.201291e-01,2.730705e-03,336.956554,0.000000e+00,9.147770e-01,9.254812e-01
1,anni_da_costruzione,-1.961296e-03,3.921133e-04,-5.001861,5.677959e-07,-2.729824e-03,-1.192768e-03
2,consumo_annuo,2.758083e-07,2.035463e-07,1.355015,1.754127e-01,-1.231350e-07,6.747516e-07
3,gg_da_installazione,4.472421e-06,1.082174e-06,4.132811,3.583538e-05,2.351399e-06,6.593444e-06
4,firmware_num,1.059749e-06,5.697811e-07,1.859923,6.289638e-02,-5.700128e-08,2.176500e-06
5,gg_da_ult_com,1.892060e-05,5.572431e-06,3.395395,6.852960e-04,7.998841e-06,2.984237e-05
6,gg_da_ult_mis,2.543134e-04,5.446088e-06,46.696529,0.000000e+00,2.436393e-04,2.649875e-04
7,costruttore_enc,1.672633e-04,1.314410e-04,1.272535,2.031832e-01,-9.035642e-05,4.248829e-04
8,modello_enc,4.685468e-03,6.379531e-05,73.445338,0.000000e+00,4.560432e-03,4.810505e-03
9,tecnologia_enc,1.378359e-03,2.148729e-04,6.414767,1.410385e-10,9.572163e-04,1.799503e-03


In [7]:
df_lett_re[COL_STATO_LETT].value_counts()
df_lett_re[COL_DIAGNOSTICA].value_counts()

NameError: name 'df_lett_re' is not defined